# Model Improvement and Selection

This notebook trains and evaluates improved machine-learning models for the Board Game Rating Predictor project.

The previous notebook established two initial models:

- a dummy baseline model
- a Linear Regression model

Linear Regression performed better than the dummy baseline, which showed that the prepared board game features contain useful predictive information.

This notebook will now try more flexible models, compare model performance, review feature importance, and select a best model for later use.

In [42]:
# Import libraries used througout this notebook
import pandas as pd

# Import Path for creating file paths that work
# reliably across different operating systems
from pathlib import Path

# Import improved regression modelling tools
# RandomForestRegressor -> tree-based regression model
# -> builds many decision trees and averages their predictions
from sklearn.ensemble import RandomForestRegressor

In [43]:
# Define current notebook folder and project rooot
current_folder = Path.cwd()
project_root = current_folder.parent

print("Current folder:", current_folder)
print("Project root:", project_root)

Current folder: c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor\notebooks
Project root: c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor


## Load Prepared Modelling Datasets

The prepared training and testing datasets are loaded from the `data/processed` folder.

These files were created during feature engineering and were also used in the baseline modelling notebook.

Using the same prepared datasets allows improved models to be compared fairly against the dummy baseline and Linear Regression models.

In [44]:
# Define paths to the prepared modelling dataset files
# -> htis will confirms notebook 6 can find processed data folder
#  before trying loading files from it
processed_data_folder = project_root / "data" / "processed"

x_train_path = processed_data_folder / "x_train_prepared.csv"
x_test_path = processed_data_folder / "x_test_prepared.csv"
y_train_path = processed_data_folder / "y_train.csv"
y_test_path = processed_data_folder / "y_test.csv"

model_feature_names_path = processed_data_folder / "model_feature_names.csv"

print("Processed data folder:", processed_data_folder)
print("Folder exists:", processed_data_folder.exists())

Processed data folder: c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor\data\processed
Folder exists: True


In [45]:
# Verify that all required prepared modelling files exist
# Saefty check
prepared_file_paths = [
    x_train_path,
    x_test_path,
    y_train_path,
    y_test_path,
    model_feature_names_path
]

for file_path in prepared_file_paths:
    print(file_path.name, "exists:", file_path.exists())

x_train_prepared.csv exists: True
x_test_prepared.csv exists: True
y_train.csv exists: True
y_test.csv exists: True
model_feature_names.csv exists: True


In [46]:
# Load prepared modelling datasets
X_train = pd.read_csv(x_train_path)
X_test = pd.read_csv(x_test_path)

y_train_data = pd.read_csv(y_train_path)
y_test_data = pd.read_csv(y_test_path)

model_feature_names = pd.read_csv(model_feature_names_path)

print("Prepared modelling datasets loaded successfully.")

Prepared modelling datasets loaded successfully.


In [47]:
# Define target column
target_column = "Rating Average"

# Convert target DataFrames into Series for modelling
# -> keeps the format consistent between other notebooks
y_train = y_train_data[target_column]
y_test = y_test_data[target_column]

print("y_train type:", type(y_train))
print("y_test type:", type(y_test))

y_train type: <class 'pandas.Series'>
y_test type: <class 'pandas.Series'>


In [48]:
# Check the shapes of the loaded modelling datasets
# -> confirmation same modelling data loaded as in notebook 5
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)
print("model_feature_names shape:", model_feature_names.shape)

X_train shape: (16273, 35)
X_test shape: (4069, 35)
y_train shape: (16273,)
y_test shape: (4069,)
model_feature_names shape: (35, 1)


In [49]:
# Verify that loaded feature columns match the saved model feature names
expected_feature_names = model_feature_names["Feature"].tolist()

x_train_columns_match = X_train.columns.tolist() == expected_feature_names
x_test_columns_match = X_test.columns.tolist() == expected_feature_names

print("X_train columns match saved feature names:", x_train_columns_match)
print("X_test columns match saved feature names:", x_test_columns_match)

X_train columns match saved feature names: True
X_test columns match saved feature names: True


In [50]:
# Check missing values in the loaded prepared datasets
# -> to cofirm feature engineering and imputation are preserved
print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())
print("Missing values in y_train:", y_train.isnull().sum())
print("Missing values in y_test:", y_test.isnull().sum())

Missing values in X_train: 0
Missing values in X_test: 0
Missing values in y_train: 0
Missing values in y_test: 0


### Prepared Modelling Dataset Load Result

The prepared modelling datasets were loaded successfully.

The training feature dataset contains 16,273 rows and 35 feature columns.

The testing feature dataset contains 4,069 rows and 35 feature columns.

The target values were converted into one-dimensional Series objects for modelling.

The loaded feature columns match the saved model feature names.

The loaded datasets contain no missing values and are ready for improved model training.

## Train Random Forest Regression Model

A Random Forest Regression model is trained as the first improved model.

Random Forest is more flexible than Linear Regression because it can learn non-linear relationships and interactions between features.

This model will be trained using the same prepared training data used by the previous models so that performance can be compared fairly.

In [51]:
# Create Random Forest Regression model
random_forest_model = RandomForestRegressor(
    # n_estimators=100 == forest will build 100 decision trees
    n_estimators=100,
    # random_state=42 == makes result reproducible
    random_state=42,
    # n_jobs=-1 == allows model to use available CPU cores
    n_jobs=-1
)

# Train Random Forest model using prepared training data
random_forest_model.fit(X_train, y_train)

print("Random Forest Regression model trained successfully.")

Random Forest Regression model trained successfully.


In [52]:
# Check the trained Random Forest model structure
print("Number of trees:", random_forest_model.n_estimators)
print("Random state:", random_forest_model.random_state)
print("Number of input features:", random_forest_model.n_features_in_)
print("Number of training features:", X_train.shape[1])

Number of trees: 100
Random state: 42
Number of input features: 35
Number of training features: 35


In [53]:
# Create Radnom Forest predictions for the training and testing feature sets
random_forest_train_predictions = random_forest_model.predict(X_train)
random_forest_test_predictions = random_forest_model.predict(X_test)

print("Random Forest train predictions shape:", random_forest_train_predictions.shape)
print("Random Forest test predictions shape:", random_forest_test_predictions.shape)

Random Forest train predictions shape: (16273,)
Random Forest test predictions shape: (4069,)


In [54]:
# Review dirstibution of Random Forest test predictions
# quick sanity check of prediction range
random_forest_test_prediction_summary = pd.Series(
    random_forest_test_predictions,
    name="Random Forest Test Predictions"
).describe()

random_forest_test_prediction_summary

count    4069.000000
mean        6.385815
std         0.708922
min         2.655400
25%         5.929200
50%         6.383100
75%         6.885100
max         8.402123
Name: Random Forest Test Predictions, dtype: float64

In [55]:
# Preview actual testing ratings beside Random Forest predictions
# -> Easy to read look at prediction
random_forest_prediction_preview = pd.DataFrame(
    {
        "Actual Rating": y_test.head(10).values,
        "Random Forest Prediction": random_forest_test_predictions[:10]
    }
)

random_forest_prediction_preview

,Actual Rating,Random Forest Prediction
0,6.23,6.52420
1,7.02,6.41930
2,6.96,7.13080
3,7.07,6.46770
4,8.88,7.66700
5,7.68,7.58775
6,6.88,7.08220
7,6.17,5.88200
8,7.53,7.07960
9,6.15,6.42940


### Random Forest Model Result

A Random Forest Regression model was trained using the prepared training dataset.

The model used 100 decision trees and all 35 prepared feature columns.

Predictions were created for both the training and testing datasets.

The Random Forest predictions will be evaluated and compared with the previous models in the next modelling step.